# ND1 final dataset — data quality report
Checks `Cleaned Data/nd1_final.csv` (KABS 1.0 + KABS 2.0 combined) for duplicates,
nulls, invalid dates, impossible counts, inconsistent syndrome flags, location gaps
and messy categories.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200, "display.max_columns", 60, "display.max_rows", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "02cleaning_scripts" else Path.cwd()
df = pd.read_csv(ROOT / "Cleaned Data" / "nd1_final.csv", low_memory=False,
                 parse_dates=["report_date", "outbreak_event_date", "submission_date"])
issues = {}  # issue name -> number of rows affected

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
print(df["source"].value_counts().to_string())

81,700 rows x 53 columns
source
KABS 1.0    80841
KABS 2.0      859


## 1. Duplicates

In [ ]:
exact = df.duplicated(keep=False)
issues["Exact duplicate rows (extra copies)"] = int(df.duplicated().sum())

# Same event reported twice: same place, day, species, disease and counts.
EVENT_KEY = ["report_date", "county", "subcounty", "ward", "locality", "species",
             "disease_condition", "number_at_risk", "number_sick", "number_dead"]
key_dupes = df.duplicated(subset=EVENT_KEY)
issues["Likely duplicate events (same key fields)"] = int(key_dupes.sum())

print("Exact duplicate rows (extra copies):", df.duplicated().sum())
print("  of which by source:", df[df.duplicated()]["source"].value_counts().to_dict())
print("Likely duplicate events on", EVENT_KEY)
print("  extra copies:", key_dupes.sum())
print("\nLargest duplicate groups:")
print(df[df.duplicated(subset=EVENT_KEY, keep=False)]
      .groupby(EVENT_KEY, dropna=False).size().sort_values(ascending=False).head(10).to_string())

Exact duplicate rows (extra copies): 31716
  of which by source: {'KABS 1.0': 31563, 'KABS 2.0': 153}
Likely duplicate events on ['report_date', 'county', 'subcounty', 'ward', 'locality', 'species', 'disease_condition', 'number_at_risk', 'number_sick', 'number_dead']
  extra copies: 34061

Largest duplicate groups:
report_date  county         subcounty      ward        locality      species  disease_condition             number_at_risk  number_sick  number_dead
2018-06-14   Kwale          Matuga         Kubo South  Village/Farm  Cattle   FMD                           3.0             5.0          0.0            111
2019-05-23   Baringo        Baringo North  Kabartonjo  Village/Farm  Sheep    Rift Valley Fever (RVF)       31.0            14.0         0.0             62
2018-11-29   Samburu        Samburu East   Waso        Village/Farm  Cattle   Foot and Mouth Disease (FMD)  3.0             5.0          0.0             59
2019-05-23   Baringo        Baringo North  Kabartonjo  Village/Far

## 2. Nulls
Some columns are expected to be mostly empty (e.g. `*_number` is only filled when the
syndrome is "Yes"; `submission_date` only exists in KABS 1.0). Core fields should be complete.

In [ ]:
nulls = pd.DataFrame({
    "nulls": df.isna().sum(),
    "pct": df.isna().mean().mul(100).round(1),
    "pct_KABS1": df[df.source == "KABS 1.0"].isna().mean().mul(100).round(1),
    "pct_KABS2": df[df.source == "KABS 2.0"].isna().mean().mul(100).round(1),
}).sort_values("pct", ascending=False)
print(nulls.to_string())

CORE = ["report_date", "outbreak_event_date", "county", "subcounty", "ward", "locality",
        "species", "disease_condition", "diagnosis_nature", "number_at_risk",
        "number_sick", "number_dead", "latitude", "longitude"]
print("\nCore fields with missing values:")
core_nulls = df[CORE].isna().sum()
print(core_nulls[core_nulls > 0].to_string())
for c, n in core_nulls[core_nulls > 0].items():
    issues[f"Missing {c}"] = int(n)

                                         nulls    pct  pct_KABS1  pct_KABS2
notes                                    81687  100.0      100.0       98.5
vaccine_batch_no                         81694  100.0      100.0       99.3
vaccination_type                         81685  100.0      100.0       98.3
bite_type                                81648   99.9      100.0       93.9
onsite                                   81006   99.2      100.0       19.2
laboratory_test_taken                    80961   99.1       99.1       98.6
gastrointestinal_bloody                  79979   97.9       97.9      100.0
death_number                             78566   96.2       96.1      100.0
animal_bites_number                      78378   95.9       96.0       93.7
sudden_death_number                      77368   94.7       94.7       94.8
neurologic_number                        76880   94.1       94.2       87.7
abortion_number                          75375   92.3       92.2       93.2
haemorrhagic

## 3. Dates

In [ ]:
today = pd.Timestamp.today().normalize()
gap = (df["report_date"] - df["outbreak_event_date"]).dt.days
date_checks = {
    "report_date in the future": df["report_date"] > today,
    "outbreak_event_date in the future": df["outbreak_event_date"] > today,
    "report_date before 2015": df["report_date"] < "2015-01-01",
    "outbreak starts after report": gap < 0,
    "outbreak > 1 year before report": gap > 365,
    "submission before report (KABS 1.0)": df["submission_date"] < df["report_date"],
}
for name, mask in date_checks.items():
    issues[name] = int(mask.sum())
    print(f"{name:40s} {mask.sum():>7,}")

print("\nReport date range by source:")
print(df.groupby("source")["report_date"].agg(["min", "max"]).to_string())
print("\nReports per year:")
print(df["report_date"].dt.year.value_counts().sort_index().to_string())

report_date in the future                      0
outbreak_event_date in the future             11
report_date before 2015                        6
outbreak starts after report               1,894
outbreak > 1 year before report              283
submission before report (KABS 1.0)           39

Report date range by source:
                min        max
source                        
KABS 1.0 2007-05-24 2026-10-16
KABS 2.0 2025-04-14 2026-10-29

Reports per year:
report_date
2007        1
2010        2
2012        2
2014        1
2017      701
2018     6753
2019     7574
2020     4074
2021    10088
2022    11228
2023    12143
2024     9147
2025    11098
2026     8888


## 4. Counts (logical checks and outliers)

In [ ]:
COUNTS = ["number_at_risk", "number_sick", "number_dead", "humans_affected_number",
          "number_slaughtered", "number_destroyed", "number_vaccinated"]
risk = df["number_at_risk"]
count_checks = {
    "negative count (any)": (df[COUNTS] < 0).any(axis=1),
    "sick > at risk": df["number_sick"] > risk,
    "dead > at risk": df["number_dead"] > risk,
    "dead + slaughtered + destroyed > at risk":
        df[["number_dead", "number_slaughtered", "number_destroyed"]].sum(axis=1, min_count=1) > risk,
    "at risk = 0 but sick/dead > 0": (risk == 0) & ((df["number_sick"] > 0) | (df["number_dead"] > 0)),
    "sick, dead and at risk all 0": (risk == 0) & (df["number_sick"] == 0) & (df["number_dead"] == 0),
}
for name, mask in count_checks.items():
    issues[name] = int(mask.sum())
    print(f"{name:42s} {mask.sum():>7,}")

print("\nDistribution (watch the max vs 99th percentile for outliers):")
print(df[COUNTS].describe(percentiles=[.5, .9, .99]).T.round(1).to_string())

# Outliers: values above 99.9th percentile AND over 1,000 animals
print("\nExtreme values (> 99.9th pct and > 1,000):")
for c in COUNTS:
    cut = max(df[c].quantile(.999), 1000)
    n = int((df[c] > cut).sum())
    if n:
        issues[f"Extreme {c} (> {cut:,.0f})"] = n
        print(f"  {c:25s} > {cut:>9,.0f}: {n} rows, max {df[c].max():,.0f}")

negative count (any)                             0
sick > at risk                               9,265
dead > at risk                                  15
dead + slaughtered + destroyed > at risk     3,711
at risk = 0 but sick/dead > 0                7,092
sick, dead and at risk all 0                 2,995

Distribution (watch the max vs 99th percentile for outliers):
                          count  mean     std  min  50%   90%    99%       max
number_at_risk          81590.0  76.3  3285.3  0.0  6.0  97.0  700.0  890000.0
number_sick             81486.0  11.7  1561.5  0.0  1.0  10.0   60.0  445000.0
number_dead             81654.0   1.4    35.9  0.0  0.0   2.0   17.0    5980.0
humans_affected_number  69425.0   0.1     4.7  0.0  0.0   0.0    2.0     980.0
number_slaughtered      81654.0   0.6    70.1  0.0  0.0   0.0    8.0   20000.0
number_destroyed        81654.0   0.4    27.8  0.0  0.0   0.0    3.0    4980.0
number_vaccinated       80975.0  14.1   212.9  0.0  0.0   0.0  257.3   30000.0

## 5. Syndrome flags vs their counts

In [ ]:
SYNDROMES = ["abortion", "sudden_death", "haemorrhagic", "neurologic", "animal_bites",
             "respiratory_signs", "oral_foot_lesions", "cutaneous_skin_lesions",
             "gastrointestinal_tract_syndromes", "other_syndromes"]
rows = []
for s in SYNDROMES:
    n = df[f"{s}_number"]
    rows.append({
        "syndrome": s,
        "yes": int((df[s] == "Yes").sum()),
        "yes_but_no_count": int(((df[s] == "Yes") & n.isna()).sum()),
        "no_but_count_gt0": int(((df[s] == "No") & (n > 0)).sum()),
        "count_gt_at_risk": int((n > risk).sum()),
    })
syn = pd.DataFrame(rows).set_index("syndrome")
print(syn.to_string())

no_syndrome = (df[SYNDROMES] != "Yes").all(axis=1)
issues["Syndrome 'Yes' but count missing"] = int(syn["yes_but_no_count"].sum())
issues["Syndrome 'No' but count > 0"] = int(syn["no_but_count_gt0"].sum())
issues["No syndrome ticked at all"] = int(no_syndrome.sum())
print(f"\nReports with no syndrome ticked: {no_syndrome.sum():,}")

                                    yes  yes_but_no_count  no_but_count_gt0  count_gt_at_risk
syndrome                                                                                     
abortion                           6320                 0                 5               283
sudden_death                       4331                 0                 1               165
haemorrhagic                       7397                 0                 8               523
neurologic                         4818                 0                 2               276
animal_bites                       3322                 0                 0               302
respiratory_signs                 28004                 0                 8              3769
oral_foot_lesions                  7884                 0                 2              1233
cutaneous_skin_lesions            12342                 0                 6              1673
gastrointestinal_tract_syndromes  19573                 0   

## 6. Location

In [ ]:
no_coords = df["latitude"].isna()
issues["Missing coordinates"] = int(no_coords.sum())
print(f"Missing coordinates: {no_coords.sum():,} ({no_coords.mean():.1%})")

# Same subcounty name appearing under more than one county
sub_multi = df.groupby("subcounty")["county"].nunique()
sub_multi = sub_multi[sub_multi > 1]
print(f"\nSubcounties listed under >1 county: {len(sub_multi)}")
for sc in sub_multi.index[:15]:
    print(f"  {sc}: {df.loc[df.subcounty == sc, 'county'].value_counts().to_dict()}")
issues["Subcounty names under >1 county"] = len(sub_multi)

# Reports far from the median point of their county (> 1.5 degrees ≈ 165 km)
centre = df.groupby("county")[["latitude", "longitude"]].transform("median")
far = ((df["latitude"] - centre["latitude"]).abs() > 1.5) | ((df["longitude"] - centre["longitude"]).abs() > 1.5)
issues["Coordinates far from their county"] = int(far.sum())
print(f"\nCoordinates > 1.5° from their county's median point: {far.sum():,}")
print(df.loc[far, "county"].value_counts().head(10).to_string())

# Many identical coordinates = default/placeholder GPS
top_pts = df.groupby(["latitude", "longitude"]).size().sort_values(ascending=False).head(5)
print("\nMost repeated coordinate pairs:")
print(top_pts.to_string())

Missing coordinates: 9,223 (11.3%)

Subcounties listed under >1 county: 0

Coordinates > 1.5° from their county's median point: 2,883
county
Marsabit           1914
Wajir               198
Garissa             124
Tana River          111
Samburu             110
Turkana             109
Elgeyo-Marakwet      38
Mombasa              33
Makueni              27
Isiolo               25

Most repeated coordinate pairs:
latitude   longitude
-0.519253  37.453860    266
-3.386665  38.539316    241
-0.519838  37.453771    206
-1.033783  39.384639    162
 3.150514  41.188077    140


## 7. Categories and free text

In [ ]:
for c in ["locality", "species", "production_system", "diagnosis_nature", "onsite"]:
    print(f"\n{c} ({df[c].nunique()} values):")
    print(df[c].value_counts(dropna=False).head(15).to_string())


def spelling_variants(series: pd.Series) -> pd.Series:
    """Group values that differ only by case, spaces or punctuation."""
    vals = series.dropna().str.split("; ").explode()
    key = vals.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
    groups = vals.groupby(key).unique()
    return groups[groups.str.len() > 1]


for c in ["disease_condition", "species", "control_methods"]:
    v = spelling_variants(df[c])
    issues[f"{c}: values with spelling variants"] = len(v)
    print(f"\n{c}: {df[c].str.split('; ').explode().nunique():,} distinct values, "
          f"{len(v)} have case/spacing variants. Examples:")
    for k in v.index[:8]:
        print("  ", list(v[k]))

junk = df["disease_condition"].str.fullmatch(r"(?i)(unknown|none|non ?specific|i don't know|other|ghhnn|n/?a)")
issues["Non-informative disease value"] = int(junk.sum())
print(f"\nNon-informative disease values (Unknown/None/I don't know/Other...): {junk.sum():,}")


locality (7 values):
locality
Village/Farm    70594
Abattoir         4698
Crush            3685
Market           1695
Clinic            892
Agrovet           117
Waterpoint         19

species (52 values):
species
Cattle     43135
Goats      19914
Sheep       7150
Dogs        3760
Camel       2653
Chicken     2644
Donkey      1099
Pigs         916
Cats         160
Bees         117
Rabbits       32
Rabbit        24
Poultry       10
Horse          8
Equine         5

production_system (4 values):
production_system
Extensive         34422
Mixed             28683
Intensive         12109
NaN                6390
Semi intensive       96

diagnosis_nature (4 values):
diagnosis_nature
Clinical                75109
Post Mortem              4976
Laboratory                753
Clinical/Post Mortem      749
NaN                       113

onsite (2 values):
onsite
NaN    81006
Yes      481
No       213

disease_condition: 2,504 distinct values, 213 have case/spacing variants. Examples:
   ['3 days S

## Summary

In [ ]:
summary = (pd.Series(issues, name="rows_affected").to_frame()
           .assign(pct_of_rows=lambda t: (t.rows_affected / len(df) * 100).round(2))
           .sort_values("rows_affected", ascending=False))
print(summary.to_string())

                                                  rows_affected  pct_of_rows
Likely duplicate events (same key fields)                 34061        41.69
Exact duplicate rows (extra copies)                       31716        38.82
sick > at risk                                             9265        11.34
Missing latitude                                           9223        11.29
Missing coordinates                                        9223        11.29
Missing longitude                                          9213        11.28
at risk = 0 but sick/dead > 0                              7092         8.68
dead + slaughtered + destroyed > at risk                   3711         4.54
sick, dead and at risk all 0                               2995         3.67
Coordinates far from their county                          2883         3.53
outbreak starts after report                               1894         2.32
Non-informative disease value                               572         0.70

In [ ]:
df.columns